In [1]:
R.version.string

[1] "R version 4.5.1 (2025-06-13 ucrt)"

In [2]:
# # 1) Pick a user-writable library path (avoids OneDrive/Documents issues)
# v <- paste(R.version$major, strsplit(R.version$minor, "\\.")[[1]][1], sep = ".")
# userlib <- normalizePath(file.path(Sys.getenv("LOCALAPPDATA"),
#                                    "R", "win-library", v),
#                          winslash = "/")
# dir.create(userlib, recursive = TRUE, showWarnings = FALSE)

# # 2) Prepend to library search path for this session
# .libPaths(c(userlib, .libPaths()))

# # 3) Install
# install.packages("intergraph")
# # sanity check
# .libPaths()


In [ ]:
library(statnet)
library(dplyr)
library(lubridate)
library(intergraph)
library(igraph)

Loading required package: tergm

Loading required package: ergm

Loading required package: network


'network' 1.19.0 (2024-12-08), part of the Statnet Project
* 'news(package="network")' for changes since last version
* 'citation("network")' for citation information
* 'https://statnet.org' for help, support, and other information



'ergm' 4.10.1 (2025-08-26), part of the Statnet Project
* 'news(package="ergm")' for changes since last version
* 'citation("ergm")' for citation information
* 'https://statnet.org' for help, support, and other information


'ergm' 4 is a major update that introduces some backwards-incompatible
changes. Please type 'news(package="ergm")' for a list of major
changes.


Loading required package: networkDynamic


'networkDynamic' 0.11.5 (2024-11-21), part of the Statnet Project
* 'news(package="networkDynamic")' for changes since last version
* 'citation("networkDynamic")' for citation information
* 'https://statnet.org' for help, support, and other informati

In [29]:
nodes <- read.csv("nodes.csv", stringsAsFactors = FALSE)
edges <- read.csv("edges.csv", stringsAsFactors = FALSE)

In [30]:
# We’ll use MBIDs as the vertex key. igraph expects a column named "name"
# in the vertices df to bind edges to vertices.
if (!"mbid" %in% names(nodes)) stop("nodes.csv must contain column 'mbid'")
if (!all(c("u","v") %in% names(edges))) stop("edges.csv must contain columns 'u' and 'v'")

nodes$name <- nodes$mbid  # key column for igraph

# --- 2) Optional helpers ------------------------------------------------------
pick_col <- function(df, ...) {
  for (nm in c(...)) if (nm %in% names(df)) return(df[[nm]])
  return(NULL)
}
`%||%` <- function(a, b) if (!is.null(a)) a else b

# --- 3) Compute/patch columns before building the graph -----------------------
# Ensure character types (avoid factors if read in elsewhere)
char_cols <- c("name","mbid","all_roles","role_major","all_genres",
               "primary_genre","first_release_date_in_window","last_release_date_in_window")
for (cc in intersect(char_cols, names(nodes))) nodes[[cc]] <- as.character(nodes[[cc]])

# Ensure numeric types
num_cols <- c("num_songs_in_window","num_collaborators_in_window","time_in_network_years")
for (nc in intersect(num_cols, names(nodes))) nodes[[nc]] <- as.numeric(nodes[[nc]])

# Standardized variants (compute if missing)
ns_std <- pick_col(nodes, "num_songs_in_window_std", "num_songs_std")
if (is.null(ns_std)) ns_std <- scale(as.numeric(nodes$num_songs_in_window))
nodes$num_songs_std <- as.numeric(ns_std)

nc_std <- pick_col(nodes, "num_collaborators_in_window_std", "num_collab_std")
if (is.null(nc_std)) nc_std <- scale(as.numeric(nodes$num_collaborators_in_window))
nodes$num_collab_std <- as.numeric(nc_std)

t_std <- pick_col(nodes, "time_in_network_years_std", "time_std")
if (is.null(t_std)) t_std <- scale(as.numeric(nodes$time_in_network_years))
nodes$time_std <- as.numeric(t_std)

# Edge attribute typing
edge_num_cols <- c("weight_raw","weight_size_adj","recency_weight",
                   "roles_overlap","genre_overlap","low_overlap")
for (ec in intersect(edge_num_cols, names(edges))) edges[[ec]] <- as.numeric(edges[[ec]])

date_cols <- c("first_collab_date","last_collab_date")
for (dc in intersect(date_cols, names(edges))) edges[[dc]] <- as.character(edges[[dc]])

# Remove self-loops and edges to missing nodes (like your keep-mask logic)
keep_u <- edges$u %in% nodes$name
keep_v <- edges$v %in% nodes$name
keep   <- keep_u & keep_v & (edges$u != edges$v)
edges  <- edges[keep, , drop = FALSE]

# --- 4) Build the igraph graph (preserves row order of edges) -----------------
# Any extra columns in `nodes` become vertex attributes.
# Any extra columns in `edges` become edge attributes.
g <- graph_from_data_frame(d = edges[, c("u","v", setdiff(names(edges), c("u","v")))],
                           directed = FALSE,
                           vertices = nodes)

# At this point:

In [31]:
V(g)$id <- V(g)$name

In [33]:
ecount(g)

# density (gden)
edge_density(g, loops = FALSE)

# Some descriptive stand-ins for ERGM terms (not a model):
# - edges term ~ ecount(g) (already above)
# - gwesp (triadic closure) → clustering/transitivity
transitivity(g, type = "global")      # global clustering coefficient
transitivity(g, type = "average")     # average local clustering
triad_census(g)                       # full triad census

# - gwdegree (degree structure) → degree stats
deg <- degree(g)
summary(deg)

[1] 7218

[1] 0.001834114

[1] 0.5396518

[1] 0.8574812

Warning message in triad_census(g):
"At vendor/cigraph/src/misc/motifs.c:1157 : Triad census called on an undirected graph. All connections will be treated as mutual."


[1] 3658125393          0   20126298          0          0          0
 [7]          0          0          0          0      35613          0
[13]          0          0          0      13916

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  0.000   2.000   4.000   5.145   7.000  93.000 

control ergm. Note that g has:

vertex attributes: role_major, primary_genre, num_songs_in_window_std, time_in_network_years_std

edge attributes: low_overlap, roles_overlap, genre_overlap, etc.

In [36]:
# # Robust core count
# ncores    <- parallel::detectCores()
# nworkers  <- if (is.na(ncores)) 1L else max(1L, ncores - 1L)

# # Choose a parallel.type only if using >1 worker
# ptype <- if (nworkers > 1L) {
#   if (.Platform$OS.type == "windows") "PSOCK" else "FORK"
# } else NULL

# set.seed(123)

# m0 <- ergm(g ~ edges + gwdegree(0.8, fixed=TRUE), estimate = "MPLE")

# # Step B: add gwesp with a conservative decay and small (even negative) start
# start <- c(
#   edges   = coef(m0)["edges"],
#   gwesp   = -0.5,                  # damp triangles to avoid blow-up at start
#   gwdegree= coef(m0)["gwdegree.decay0.8"] %||% 0  # if absent, fallback 0
# )

# ctrl <- control.ergm(
#   init.method         = "MPLE",
#   MCMLE.density.guard = 100,       # bump a bit, but don’t rely on this
#   MCMC.prop.weights   = "TNT",
#   MCMC.burnin         = 5e4,
#   MCMC.interval       = 1e3,
#   MCMC.samplesize     = 2e3,
#   parallel          = nworkers,
#   parallel.type     = ptype,   # NULL if single-core
# )


In [ ]:
# ctrl_fast <- control.ergm(
#   init.method       = "MPLE",
#   MCMC.prop.weights = "TNT",
#   # lighter sampler to get you moving
#   MCMC.burnin       = 10000,
#   MCMC.interval     = 200,
#   MCMC.samplesize   = 1000,
#   # cap how long MCMLE keeps iterating
#   MCMLE.maxit       = 10,
#   # parallel: avoid PSOCK overhead on Windows
#   parallel          = if (.Platform$OS.type == "windows") 1L else nworkers,
#   parallel.type     = if (.Platform$OS.type == "windows") NULL else "FORK"
# )

simple ERGM, every dyad has the same probability of collaboration independent of anything else. 

H: is the network denser or sparser than pure randomness?

Single edges reflect baseline log odds of a tie, if edges = -4, each potential pair has exp(-4) = 0.018 probability of collaborating

In [ ]:
# theta0 <- c(qlogis(gden(g)), -0.5, -0.5)
# sims <- simulate(g ~ edges + gwesp(0.5, fixed=TRUE) + gwdegree(1.5, fixed=TRUE),
#                  coef = theta0, nsim = 20, output = "stats")
# summary(sims[,"edges"])   # should be in the same ballpark as ~7

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   4741    4810    4898    4935    5010    5286 

geometrically weighted edgewise shared partners captures triadict closure (friends of friends tend to collaborate). A positive value will indicate strong clustering

gwdegress: geometrically weighted degree models skew of degree distribution (preferential attachment). A positive value will indicate that popular artists attract many collaborators

H: does collaboration cluster into triangles? do hubs form?

In [37]:
net <- asNetwork(g) 

In [ ]:
# m1 <- ergm(
#   g ~ edges + gwesp(0.5, fixed=TRUE) + gwdegree(0.8, fixed=TRUE),
#   control = ctrl
# )
# https://chatgpt.com/c/6913a752-f200-8327-85ec-e8949376ab0c
m1_fast <- ergm(net ~ edges + gwesp(0.5, fixed=TRUE) + gwdegree(0.8, fixed=TRUE), estimate="MPLE")
summary(m1_fast)

Starting maximum pseudolikelihood estimation (MPLE):

Obtaining the responsible dyads.

Evaluating the predictor and response matrix.

Maximizing the pseudolikelihood.

Finished MPLE.

Evaluating log-likelihood at the estimate. 




In [42]:
g_names   <- igraph::V(g)$name
deg_obs   <- igraph::degree(g, mode = "all")
tri_obs   <- igraph::count_triangles(g, vids = igraph::V(g))

In [44]:
nsim <- 100
sims <- simulate(
  m1_fast,
  nsim      = nsim,
  output    = "network",
  control   = control.simulate.ergm(
    MCMC.burnin  = 1e5,
    MCMC.interval= 1e3
  )
)

In [45]:
deg_acc <- numeric(length(g_names))
tri_acc <- numeric(length(g_names))

for (s in seq_len(nsim)) {
  net_s <- sims[[s]]
  g_s   <- asIgraph(net_s)
  g_s   <- igraph::simplify(g_s, remove.multiple = TRUE, remove.loops = TRUE)

  # Ensure vertex order aligns by name
  # (asIgraph should preserve order, but we make it explicit & robust)
  sim_names <- igraph::V(g_s)$name
  m <- match(g_names, sim_names)

  # Some simulations might drop isolated vertices if names are missing; guard that:
  # (If any NA in m, replace with 0-length or 0 stats.)
  d_s <- numeric(length(g_names))
  t_s <- numeric(length(g_names))

  d_s[!is.na(m)] <- igraph::degree(g_s, mode = "all")[m[!is.na(m)]]
  t_s[!is.na(m)] <- igraph::count_triangles(g_s, vids = igraph::V(g_s))[m[!is.na(m)]]

  deg_acc <- deg_acc + d_s
  tri_acc <- tri_acc + t_s
}

In [ ]:
deg_exp <- deg_acc / nsim
tri_exp <- tri_acc / nsim

In [48]:
node_summary <- data.frame(
  node            = g_names,
  degree_obs      = as.numeric(deg_obs),
  degree_exp      = as.numeric(deg_exp),
  tri_obs         = as.numeric(tri_obs),
  tri_exp         = as.numeric(tri_exp),
  residual_degree = as.numeric(deg_obs - deg_exp),
  residual_tri    = as.numeric(tri_obs - tri_exp),
  stringsAsFactors = FALSE
)

In [49]:
head(node_summary)

,node,degree_obs,degree_exp,tri_obs,tri_exp,residual_degree,residual_tri
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,777a21a8-0d0f-4cf3-86b4-65bc0eba5649,2,0,1,0,2,1
2,703c557f-82bb-4646-ae2f-b5a3ef3f6148,12,0,66,0,12,66
3,25848dee-8562-4a78-b375-3a80b61da629,5,0,10,0,5,10
4,3e7bcc53-53d4-41d8-afbc-69f82b858fd9,6,0,15,0,6,15
5,43a0ce3a-a3d5-45c0-b944-bd8f5e021237,5,0,4,0,5,4
6,89aa5ecb-59ad-46f5-b3eb-2d424e941f19,23,0,56,0,23,56


In [ ]:
# https://chatgpt.com/share/e/6913d3f1-55f8-800a-8685-df369efa7346
fast_node_gwesp_score <- function(g, decay = 0.5) {
  gi <- g
  # shared partners per edge
  el <- as_edgelist(gi, names = FALSE)
  A <- as_adj(gi, sparse = TRUE)
  sp <- function(i, j) sum(A[i, ] & A[j, ])  # shared partners count

  # geometric weight with diminishing returns
  # NOTE: This mirrors the idea of GWESP weighting (more SP -> higher, but with diminishing gains).
  # You don't need the exact internal ERGM transform to get a useful ranking score.
  w_from_s <- function(s, decay) 1 - exp(-decay * s)

  edge_w <- numeric(nrow(el))
  for (e in seq_len(nrow(el))) {
    s_ij <- sp(el[e, 1], el[e, 2])
    edge_w[e] <- w_from_s(s_ij, decay)
  }

  n <- vcount(gi)
  node_w <- numeric(n)
  for (e in seq_len(nrow(el))) {
    i <- el[e, 1]; j <- el[e, 2]
    node_w[i] <- node_w[i] + edge_w[e]
    node_w[j] <- node_w[j] + edge_w[e]
  }

  data.frame(
    node = V(gi)$name %||% as.character(seq_len(n)),
    gwesp_like_node_score = node_w,
    stringsAsFactors = FALSE
  )
}
gwesp_contribs <- fast_node_gwesp_score(g)

Warning message:
"`as_adj()` was deprecated in igraph 2.1.0.
ℹ Please use `as_adjacency_matrix()` instead."


In [53]:
head(gwesp_contribs)

,node,gwesp_like_node_score
,<chr>,<dbl>
1,777a21a8-0d0f-4cf3-86b4-65bc0eba5649,0.7869387
2,703c557f-82bb-4646-ae2f-b5a3ef3f6148,11.9509587
3,25848dee-8562-4a78-b375-3a80b61da629,4.3233236
4,3e7bcc53-53d4-41d8-afbc-69f82b858fd9,5.5074900
5,43a0ce3a-a3d5-45c0-b944-bd8f5e021237,2.6833004
6,89aa5ecb-59ad-46f5-b3eb-2d424e941f19,19.1703870


# Final frame creation
figured out how to get most features, so now we create

In [74]:
clust_local <- transitivity(g, type = "local", isolates = "zero") # clustering

In [75]:
triangles_per_node <- count_triangles(g)  # triange count

In [76]:
deg <- degree(g, mode = "all")
open_wedges <- choose(deg, 2) - triangles_per_node
open_wedges[deg < 2] <- 0 # Open wedges (two-paths that are not closed) (per node)

In [77]:
gwesp_node_score <- function(gi, decay = 0.5) {
  A  <- as_adj(gi, sparse = TRUE)           # 0/1 adjacency
  SP <- A %*% A                              # shared partners between i and j
  el <- as_edgelist(gi, names = FALSE)

  # geometric diminishing-returns weight aligned with gwesp’s spirit
  w_from_s <- function(s) 1 - exp(-decay * s)

  edge_w <- numeric(nrow(el))
  for (e in seq_len(nrow(el))) {
    s_ij <- SP[el[e,1], el[e,2]]
    edge_w[e] <- w_from_s(s_ij)
  }
  node_w <- numeric(vcount(gi))
  for (e in seq_len(nrow(el))) {
    i <- el[e,1]; j <- el[e,2]
    node_w[i] <- node_w[i] + edge_w[e]
    node_w[j] <- node_w[j] + edge_w[e]
  }
  node_w
}

gwesp_score <- gwesp_node_score(g, decay = 0.5)

In [78]:
deg <- degree(g, mode = "all")

In [79]:
eig_cen  <- eigen_centrality(g)$vector
pagerank <- page_rank(g)$vector
kcore    <- coreness(g)
betw     <- betweenness(g, directed = is.directed(g), normalized = TRUE)
close    <- closeness(g, normalized = TRUE)

In [ ]:
gi <- g

# (optional) guardrail for closeness on isolates
close[!is.finite(close)] <- 0

# pull ids
nodes <- if (!is.null(igraph::V(gi)$name)) igraph::V(gi)$name else as.character(seq_len(igraph::vcount(gi)))

# assemble
node_features <- data.frame(
  mbid              = nodes,
  clustering_local  = as.numeric(clust_local),
  triangles         = as.numeric(triangles_per_node),
  open_wedges       = as.numeric(open_wedges),
  gwesp_contrib     = as.numeric(gwesp_contribs),
  degree            = as.numeric(deg),
  eigencentrality   = as.numeric(eig_cen),
  pagerank          = as.numeric(pagerank),
  kcore             = as.numeric(kcore),
  betweenness       = as.numeric(betw),
  closeness         = as.numeric(close),
  same_genre_share  = as.numeric(same_genre_share),
  stringsAsFactors  = FALSE
)

ERROR: Error: nrow(node_features) == igraph::vcount(gi) is not TRUE


In [85]:
head(node_features)

,mbid,clustering_local,triangles,open_wedges,gwesp_contrib,degree,eigencentrality,pagerank,kcore,betweenness,closeness,same_genre_share
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,777a21a8-0d0f-4cf3-86b4-65bc0eba5649,1.0000000,1,0,0,2,2.509173e-05,0.0001814516,2,0.000000000,0.1492665,1.0000000
2,703c557f-82bb-4646-ae2f-b5a3ef3f6148,1.0000000,66,0,0,12,1.472325e-04,0.0004127831,12,0.000000000,0.1707496,0.9166667
3,25848dee-8562-4a78-b375-3a80b61da629,1.0000000,10,0,0,5,0.000000e+00,0.0002406972,5,0.000000000,0.4303797,1.0000000
4,3e7bcc53-53d4-41d8-afbc-69f82b858fd9,1.0000000,15,0,0,6,5.511010e-03,0.0002849998,6,0.000000000,0.2302610,0.5000000
5,43a0ce3a-a3d5-45c0-b944-bd8f5e021237,0.4000000,4,6,0,5,5.220216e-06,0.0003960507,3,0.002117678,0.1470930,0.4000000
6,89aa5ecb-59ad-46f5-b3eb-2d424e941f19,0.2213439,56,197,0,23,1.426881e-02,0.0011203706,9,0.004394862,0.2373170,0.1304348


In [90]:
write.csv(node_features, "node_features.csv", row.names = FALSE)